In [12]:
import numpy as np
import uproot as ur
import awkward as ak
import matplotlib.pyplot as plt
import mplhep as hep
hep.style.use("CMS")
hep.style.use(hep.style.CMS)
import matplotlib.colors as mcolors
from scipy.optimize import curve_fit
import time
import pandas as pd
from mpl_toolkits.axes_grid1 import make_axes_locatable

from cycler import cycler
plt.rcParams["axes.prop_cycle"] = cycler('color', ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'])

In [13]:
is_simulation = False

if is_simulation:
    data_directory = "/media/miguel/Elements_2024/CLAS_data/"
    file_name = "ntuples_LD2Csolid_clasdis_deuteron_3000files_roottuples.root"
else:
    data_directory = "/media/miguel/Elements_2024/CLAS_data/"
    file_name = "ntuples_020131-020176_pass08_withcharge.root"

apply_fiducial_cuts = True
target_vertex_selection = False
solid_name = "C"
num_sectors = 6
make_event_builder_plots = False
E_beam = 10.547

In [14]:
from concurrent.futures import ThreadPoolExecutor

branches_to_open = ['pid', 'chi2pid', 'p_x', 'p_y', 'p_z', 'p', 'status', 'v_x', 'v_y', 'v_z', 'theta', 'phi', 'track_charge', 'sector', 'NDF', 'chi2', 'E_PCAL', 'E_ECIN', "E_ECOUT", 'PCAL_U', 'PCAL_V', 'PCAL_W', 'Nphe_HTCC', 'DC_region1_x', 'DC_region1_y', 'DC_region1_z', 'DC_region1_edge', 'DC_region2_x', 'DC_region2_y', 'DC_region2_z', 'DC_region2_edge', 'DC_region3_x', 'DC_region3_y', 'DC_region3_z', 'DC_region3_edge']

def load_branch(branch_name, events):
    return events[branch_name].array()

start_time = time.time()
with ur.open(f"{data_directory+file_name}:data") as events:
    print(f"Took {time.time() - start_time:.2f} s to open file")
    start_time = time.time()
    
    # Use ThreadPoolExecutor to load branches in parallel
    with ThreadPoolExecutor(max_workers=len(branches_to_open)) as executor:
        results = list(executor.map(lambda b: load_branch(b, events), branches_to_open))

    branches = ak.Array(dict(zip(branches_to_open, results)))
    print(f"Took {time.time() - start_time:.2f} s to load branches")

In [17]:
with ur.open(f"{data_directory+file_name}:meta") as meta:
    meta_info = meta.arrays()

In [18]:
if is_simulation:
    plot_title = "RGE LD2 + C: clasdis simulation solid"
else:
    plot_title = "RGE LD2 + C: 20131-20176 pass 0.8"
print(plot_title)

RGE LD2 + C: 20131-20176 pass 0.8


### Each data files contain all the particles across the events. The particles are not separated into different events (i.e. they are flattened)
### We want to select particles that were identified as trigger electrons by the reconstruction software (event builder) and then apply some additional cuts to further refine the selection.
The cuts that were already applied by the event builder are:
- At least two photoelectrons in the HTCC
- 60 MeV in the PCAL
- 5 sigma cut on SF vs p

The list of cuts we want to apply (motivated by RGA and RGM analysis notes) are:
- $Q^2 > 1~ \rm{GeV}^2$
- $W > 1.7 ~\rm{GeV}$
- $2~\rm{GeV} < p_{elec} < 8~\rm{GeV}$
- $\theta_{elec}>5^{\circ}$
- $y < 0.8$
- Fiducial cuts
- Partial sampling fraction cut
- SF vs Epcal cut

### Calculating the charge in the files

In [19]:
run_numbers = np.unique(meta_info["run_number"])

In [24]:
print(run_numbers)

[20131, 20132, 20133, 20134, 20135, ..., 20172, 20173, 20174, 20175, 20176]


In [21]:
total_charge = 0
total_num_events = 0
total_num_events_skipped = 0

for run in run_numbers:
    run_data = meta_info[meta_info["run_number"]==run]
    events_in_run = np.sort(np.unique(run_data["event_number"]))
    total_num_events += len(events_in_run)
    data_with_charge = run_data[run_data["fcupgated"]>-1]
    fcupgated_data = data_with_charge["fcupgated"]/1000
    
    max_charge = max(fcupgated_data)
    max_index = ak.argmax(fcupgated_data)
    max_charge_event = data_with_charge["event_number"][max_index]
    num_events_after_max = len(events_in_run[events_in_run>max_charge_event])
    
    min_charge = min(fcupgated_data)
    min_index = ak.argmin(fcupgated_data)
    min_charge_event = data_with_charge["event_number"][min_index]
    num_events_before_min = len(events_in_run[events_in_run<min_charge_event])

    total_num_events_skipped += (num_events_after_max + num_events_before_min)
    total_charge += (max_charge - min_charge)
    print(f"Run #{run} has {max_charge-min_charge} microC across {len(events_in_run)}, with {(num_events_after_max + num_events_before_min)} events skipped")

Run #20131 has 0.7780517578125004 microC across 658347, with 189 events skipped
Run #20132 has 0.8324096679687498 microC across 661000, with 291 events skipped
Run #20133 has 0.8039731445312506 microC across 658566, with 327 events skipped
Run #20134 has 0.8007358398437501 microC across 654729, with 334 events skipped
Run #20135 has 0.5518461914062502 microC across 473811, with 312 events skipped
Run #20136 has 0.7959511718749996 microC across 651002, with 384 events skipped
Run #20137 has 0.7818781738281251 microC across 644975, with 297 events skipped
Run #20138 has 0.7949370117187495 microC across 648380, with 262 events skipped
Run #20139 has 0.7054218749999999 microC across 631043, with 359 events skipped
Run #20140 has 0.7874072265624998 microC across 648311, with 53 events skipped
Run #20141 has 0.8483637695312503 microC across 674161, with 326 events skipped
Run #20143 has 0.7774421386718751 microC across 653554, with 397 events skipped
Run #20144 has 0.8428706054687498 microC 

In [23]:
print(f"Total charge being considered is {round(total_charge/1000, 5)} mC across {total_num_events} events")
print(f"We skipped { round(total_num_events_skipped/total_num_events*100,2)}% events while calculating")

Total charge being considered is 0.03206 mC across 26024189 events
We skipped 0.05% events while calculating


### Selecting trigger electrons

In [ ]:
print(f"Number of particles: {len(branches['pid'])}")

In [ ]:
print(len(branches['pid']))

In [ ]:
event_builder_mask = (branches["pid"]==11)
event_builder_electrons = branches[event_builder_mask]
original_event_builder_electrons = branches[event_builder_mask]
print(f"Percent of particles remaining after EB PID cut: {round(len(event_builder_electrons['pid'])/len(branches['pid']), 3)*100}")
status_mask = (event_builder_electrons["status"] > -4000) & (event_builder_electrons["status"] <= -2000)
print(f"Percent of EB electrons remaining after status cut: {round(len(event_builder_electrons[status_mask]['pid'])/len(original_event_builder_electrons['pid']), 3)*100}")
event_builder_electrons = event_builder_electrons[status_mask]
# Defining a sampling fraction variable to make life easier later
event_builder_electrons["SF"] = (event_builder_electrons["E_PCAL"] + event_builder_electrons["E_ECOUT"] + event_builder_electrons["E_ECIN"])/event_builder_electrons["p"]

### Calculating DIS quantities for trigger electrons

In [ ]:
def calc_theta_lab(px, py, pz):
    return np.arctan2(np.sqrt(px**2 + py**2), pz)
def calc_Q2(p, beam_E, theta):
    return 4 * p * beam_E* np.sin(theta/2)*np.sin(theta/2)
def calc_nu(p, beam_E):
    return beam_E - p
def calc_xb(Q2, beam_E, nu):
    proton_mass = .938
    return np.array(Q2/(2*proton_mass*nu))
def calc_y(p, beam_E):
    return calc_nu(p, beam_E)/beam_E
def calc_W2(p, beam_E, theta):
    proton_mass = .938
    return proton_mass*proton_mass + 2*proton_mass*calc_nu(p, beam_E) - calc_Q2(p, beam_E, theta)

### Applying DIS cuts, theta cut, and p cut

In [ ]:
plt.hist(event_builder_electrons["chi2pid"], bins=50)

In [ ]:
event_builder_electrons["theta"] = calc_theta_lab(event_builder_electrons["p_x"], event_builder_electrons["p_y"], event_builder_electrons["p_z"])
event_builder_electrons["theta_degrees"] = event_builder_electrons["theta"]*180/np.pi
event_builder_electrons["Q2"] = calc_Q2(event_builder_electrons["p"], E_beam, event_builder_electrons["theta"])
event_builder_electrons["nu"] = calc_nu(event_builder_electrons["p"], E_beam)
event_builder_electrons["x"] = calc_xb(event_builder_electrons["Q2"], E_beam, event_builder_electrons["nu"])
event_builder_electrons["y"] = calc_y(event_builder_electrons["p"], E_beam)
event_builder_electrons["W"] = np.sqrt(calc_W2(event_builder_electrons["p"], E_beam, event_builder_electrons["theta"]))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(12, 10))
ax = plt.gca()

# EIC sqrt s = 89 GeV
s = 89**2 
y_min = 0.01
y_max = .95
x = np.logspace(-4, 0, 500)
Q2_min = x * y_min * s
Q2_max = x * y_max * s
plt.fill_between(x, Q2_min, Q2_max, color='#1f77b4', alpha=0.6, label="EIC $\sqrt{s} = 89$ GeV\n (up to A = 238)")
plt.plot(x, Q2_min, color='#1f77b4')
plt.plot(x, Q2_max, color='#1f77b4')

# EIC sqrt s = 20 GeV
s = 20**2 
Q2_min = x * y_min * s
Q2_max = x * y_max * s
plt.fill_between(x, Q2_min, Q2_max, color='#9467bd', alpha=0.6, label="EIC $\sqrt{s} = 20$ GeV\n (up to A = 238)")
plt.plot(x, Q2_min, color='#1f77b4')
plt.plot(x, Q2_max, color='#1f77b4')

hermes_x = [.0125, .0175, .025, .035, .045, .055, .07, .09, .125, .175, .25, .35]
hermes_Q2 = [
    np.logspace(np.log10(0.3),  np.log10(0.5),  4),
    np.logspace(np.log10(0.45), np.log10(0.8),  5),
    np.logspace(np.log10(0.5),  np.log10(1.1),  8),
    np.logspace(np.log10(0.7),  np.log10(1.8),  10),
    np.logspace(np.log10(0.9),  np.log10(2),    10),
    np.logspace(np.log10(0.9),  np.log10(2.5),  10),
    np.logspace(np.log10(1),    np.log10(3),    10),
    np.logspace(np.log10(1.1),  np.log10(3.5),  10),
    np.logspace(np.log10(1.2),  np.log10(4),    10),
    np.logspace(np.log10(1.3),  np.log10(6),    10),
    np.logspace(np.log10(1.4),  np.log10(10),   12),
    np.logspace(np.log10(1.6),  np.log10(12),   12)
]

x_vals = []
q2_vals = []

for x, q2_range in zip(hermes_x, hermes_Q2):
    x_vals.extend([x] * len(q2_range))
    q2_vals.extend(q2_range)

plt.scatter(event_builder_electrons["x"], event_builder_electrons["Q2"], label="JLab RGE $\sqrt{s} = 4.5$ GeV\n(D, C, Al, Cu, Sn, Pb) ", color='#ff7f0e')
jlab_6gev_data=np.array([[3.500E-1, 3.266E+0, 8.625E-1, 1.006E+0, 1.034E-2, 1.143E-2],
[3.750E-1, 3.466E+0, 8.541E-1, 9.807E-1, 8.507E-3, 1.105E-2],
[4.000E-1, 3.661E+0, 8.459E-1, 9.751E-1, 9.586E-3, 1.089E-2],
[4.250E-1, 3.853E+0, 8.378E-1, 9.721E-1, 6.011E-3, 1.077E-2],
[4.500E-1, 4.041E+0, 8.299E-1, 9.778E-1, 8.015E-3, 1.075E-2],
[4.750E-1, 4.225E+0, 8.221E-1, 9.779E-1, 7.579E-3, 1.076E-2],
[5.000E-1, 4.406E+0, 8.145E-1, 9.541E-1, 5.835E-3, 1.077E-2],
[5.250E-1, 4.584E+0, 8.070E-1, 9.361E-1, 5.704E-3, 1.077E-2],
[5.500E-1, 4.759E+0, 7.997E-1, 9.347E-1, 5.966E-3, 1.086E-2],
[5.750E-1, 4.930E+0, 7.924E-1, 9.250E-1, 7.195E-3, 1.079E-2],
[6.000E-1, 5.098E+0, 7.853E-1, 9.244E-1, 5.292E-3, 1.076E-2],
[6.250E-1, 5.264E+0, 7.784E-1, 9.244E-1, 6.667E-3, 1.070E-2],
[6.500E-1, 5.426E+0, 7.715E-1, 9.023E-1, 9.596E-3, 1.039E-2],
[6.750E-1, 5.586E+0, 7.648E-1, 9.071E-1, 8.462E-3, 1.040E-2],
[7.000E-1, 5.743E+0, 7.582E-1, 9.047E-1, 9.134E-3, 1.032E-2],
[7.250E-1, 5.897E+0, 7.517E-1, 8.985E-1, 1.006E-2, 1.021E-2],
[7.500E-1, 6.049E+0, 7.453E-1, 9.120E-1, 6.692E-3, 1.032E-2],
[7.750E-1, 6.198E+0, 7.391E-1, 9.301E-1, 1.067E-2, 1.050E-2],
[8.000E-1, 6.344E+0, 7.329E-1, 9.367E-1, 1.169E-2, 1.053E-2],
[8.250E-1, 6.488E+0, 7.268E-1, 9.833E-1, 1.349E-2, 1.103E-2],
[8.500E-1, 6.630E+0, 7.209E-1, 1.020E+0, 1.552E-2, 1.142E-2],
[8.750E-1, 6.769E+0, 7.150E-1, 1.123E+0, 1.589E-2, 1.254E-2],
[9.000E-1, 6.907E+0, 7.092E-1, 1.169E+0, 2.385E-2, 1.303E-2],
[9.250E-1, 7.042E+0, 7.035E-1, 1.277E+0, 1.887E-2, 1.421E-2],
[9.500E-1, 7.174E+0, 6.979E-1, 1.361E+0, 4.168E-2, 1.512E-2]])
jlab_6gev_x = jlab_6gev_data[:,0]
jlab_6gev_Q2 = jlab_6gev_data[:,1]
#sqrt s = 3.4 GeV
plt.scatter(jlab_6gev_x, jlab_6gev_Q2, color='#d62728', marker='s', label='JLab Hall C $\sqrt{s} = 3.4$ GeV\n($^1H$, D, $^3He$, $^4He$, Be, C, Cu, Au)')

#sqrt s = 7.2 GeV
plt.scatter(x_vals, q2_vals, color='#2ca02c', marker='^', label='HERMES $\sqrt{s} = 7.2$ GeV\n(D, $^3He$, N)')




box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width * 0.75, box.height])  # shrink width to 75%

ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), handletextpad=0.4)

# Final formatting
ax.set_yscale('log')
# ax.set_xscale('log')
ax.set_xlim(0.001, 0.8)
ax.set_ylim(1, 50)
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$Q^2$ (GeV$^2$)')
ax.set_title("e+A DIS phase-space")


In [ ]:
electron_cuts = (event_builder_electrons["Q2"] > 1) & (event_builder_electrons["W"] > 2) & (event_builder_electrons["y"] < .8) & (event_builder_electrons["p"]>2) & (event_builder_electrons["p"]< 8) & (event_builder_electrons["theta_degrees"]>5)
print(f"Percent of EB electrons remaining after DIS cuts: {round(len(event_builder_electrons[electron_cuts]['pid'])/len(original_event_builder_electrons['pid']), 3)*100}")
electrons = event_builder_electrons[electron_cuts]

In [ ]:
def Q2_calculation_from_W(W_val, x_values):
    Mp = 0.938272
    x_values = np.asarray(x_values)
    numerator = x_values*(Mp*Mp - W_val * W_val)
    denominator = x_values - 1
    return np.divide(numerator, denominator)
def Q2_calculation_from_y(y_val, x_values):
    Eb = 10.547
    Mp = 0.938272
    return 2*Mp*Eb*y_val*np.asarray(x_values)

def Q2_calculation_from_theta(theta_val, x_val):
    Eb = 10.547
    Mp = 0.938272
    return 4*Eb*np.sin(theta_val/2)**2/(1+2/(x_val*Mp))

In [ ]:
fig = plt.figure(figsize=(10,8))

H, xedges, yedges, _ = plt.hist2d(event_builder_electrons["x"].to_numpy(), event_builder_electrons["Q2"].to_numpy(), bins=(250,250), range=[(0.01,1),(0,12)], norm=mcolors.LogNorm());
plt.colorbar()
plt.xlabel("x")
plt.ylabel("$Q^{2} ~(GeV^{2})$")
plt.title(f"{plot_title}")
x_values = np.linspace(0, 1, num=100)  # Linear spacing between 0 and 1
# Calculate Q2 values

plt.plot(x_values, Q2_calculation_from_W(2., x_values), label = "W = 2 GeV", color='red')
plt.plot(x_values, Q2_calculation_from_y(.8, x_values), label = "y = .8", color='black')
plt.plot(x_values, np.ones(len(x_values))*1, label = "$Q^2 = 1~ GeV^2$", color='orange')
plt.legend()

In [ ]:
fig = plt.figure(figsize=(10,8))

H, xedges, yedges, _ = plt.hist2d(electrons["x"].to_numpy(), electrons["Q2"].to_numpy(), bins=(250,250), range=[(0.01,1),(0,12)], norm=mcolors.LogNorm());
plt.colorbar()
plt.xlabel("x")
plt.ylabel("$Q^{2} ~(GeV^{2})$")
plt.title(f"{plot_title}")
x_values = np.linspace(0, 1, num=100)  # Linear spacing between 0 and 1
# Calculate Q2 values

plt.plot(x_values, Q2_calculation_from_W(2., x_values), label = "W = 2 GeV", color='red')
plt.plot(x_values, Q2_calculation_from_y(.8, x_values), label = "y = .8", color='black')
plt.plot(x_values, np.ones(len(x_values))*1, label = "$Q^2 = 1 ~GeV^2$", color='orange')
plt.legend()

### Charge of tracks

In [ ]:
if make_event_builder_plots:
    plt.hist(branches["track_charge"], bins=3, range=(-1.5,1.5))
    plt.xlabel("Track charge")
    plt.title(plot_title)
track_charge_mask = branches["track_charge"]==-1

In [ ]:
if make_event_builder_plots:
    print(f"When we require the track charge to be negative, we keep {round(len(branches['pid'][track_charge_mask])/len(branches['pid']),3)*100}% of particles")

### Number of photoelectrons in HTCC
### Charged pions create 2 or fewer photoelectrons and can use that as a veto

In [ ]:
if make_event_builder_plots:
    plt.hist(np.array(branches["Nphe_HTCC"][track_charge_mask]), bins = 40, range=(0,40), histtype='step')
    plt.xlabel("HTCC $N_{PE}$")
    plt.title(plot_title+"\n Negative tracks")
    plt.yscale('log')
    plt.axvline(x=2, color='red', linewidth=2)
    HTCC_mask = np.array(branches["Nphe_HTCC"][track_charge_mask])>2
    HTCC_mask_nocharge = np.array(branches["Nphe_HTCC"])>2

In [ ]:
if make_event_builder_plots:
    print(f"When we require more than 2 photoelectrons in HTCC, we keep {round(len(branches['pid'][track_charge_mask][HTCC_mask])/len(branches['pid']),3)*100}% of particles")
    print(f"When we require more than 2 photoelectrons in HTCC, we keep {round(len(branches['pid'][HTCC_mask_nocharge])/len(branches['pid']),3)*100}% of negative particles")

### Status cuts
### The electron is restricted to being in the Forward Detector
### Particles must have status from (-4000, -2000]

In [ ]:
status_mask = (branches["status"] > -4000) & (branches["status"] <= -2000)
if make_event_builder_plots:
    plt.hist(np.array(branches["status"]), bins = 40)
    plt.xlabel("Status")
    plt.title(plot_title)
    plt.yscale('log')
# status_mask_with_charge = (branches["status"][track_charge_mask] > -4000) & (branches["status"][track_charge_mask] <= -2000)
# print(f"When we require the status range, we keep {round(len(branches['pid'][track_charge_mask][status_mask_with_charge])/len(branches['pid']),3)*100}% of particles")
print(f"When we require the status range, we keep {round(len(branches['pid'][status_mask])/len(branches['pid']),3)*100}% of particles")

### PCAL energy
### Can remove negative pions using the PCAL energy since they should behave like MIPs, i.e. deposit little energy

In [ ]:
if make_event_builder_plots:
    plt.hist2d(
        np.array(branches["E_PCAL"][track_charge_mask]), 
        np.array(branches["E_ECIN"][track_charge_mask]+branches["E_ECOUT"][track_charge_mask]),
        bins=(250,250),
        range=[(0,.6), (0,.6)],
        norm=mcolors.LogNorm());
    plt.xlabel("$E_{PCAL}$ (GeV)")
    plt.ylabel("$E_{ECIN}+E_{ECOUT}$ (GeV)")
    plt.colorbar()
    plt.title(plot_title+"\n Negative tracks")
    plt.axvline(x=0.06, color='red', linewidth=2)
    PCAL_mask = np.array(branches["E_PCAL"][track_charge_mask])>.06
    PCAL_mask_nocharge = np.array(branches["E_PCAL"])>.06

In [ ]:
if make_event_builder_plots:
    print(f"When we require more than 60 MeV in the PCAL, we keep {round(len(branches['pid'][track_charge_mask][PCAL_mask])/len(branches['pid']),3)*100}% of particles")
    print(f"When we require more than 60 MeV in the PCAL, we keep {round(len(branches['pid'][PCAL_mask_nocharge])/len(branches['pid']),3)*100}% of negative particles")

### Confirming the pid==11 has the previously mentioned cuts

In [ ]:
if make_event_builder_plots:
    HTCC_test_mask = (branches['Nphe_HTCC'][branches['pid']==11]>=2)
    print(f"{len(branches['Nphe_HTCC'][branches['pid']==11][HTCC_test_mask])/len(branches['Nphe_HTCC'][branches['pid']==11])*100}% of electron candidates satisfy HTCC cuts")
    
    PCAL_test_mask = (branches['E_PCAL'][branches['pid']==11]>=.06)
    print(f"{len(branches['E_PCAL'][branches['pid']==11][PCAL_test_mask])/len(branches['E_PCAL'][branches['pid']==11])*100}% of electron candidates satisfy PCAL cuts")
    
    track_test_mask = branches['track_charge'][branches['pid']==11]==-1
    print(f"{len(branches['track_charge'][branches['pid']==11][track_test_mask])/len(branches['track_charge'][branches['pid']==11])*100}% of electron candidates satisfy track cuts")
    
    status_test_mask = (branches['status'][branches['pid']==11]>-4000) & (branches['status'][branches['pid']==11]<=-2000)
    print(f"{len(branches['status'][branches['pid']==11][status_test_mask])/len(branches['status'][branches['pid']==11])*100}% of electron candidates satisfy status cuts")
    print("Should apply status mask in addition to pid==11 mask since this is not 100%")

In [ ]:
print(f"When we require event builder electrons, we keep {round(len(branches['pid'][branches['pid']==11])/len(branches['pid']),3)*100}% of particles")

In [ ]:
# print(f"When we require the status range, we keep {round(len(branches['pid'][(electron_pid_mask) & (status_mask)])/len(branches['pid'][electron_pid_mask]),3)*100}% of EB electrons")

### Applying fiducial cuts
These cuts are the following:
- $V_{PCAL}>14$ cm
- $W_{PCAL}>14$ cm
- $\chi^2$/NDF cuts for DC

In [ ]:
low_bin, high_bin, num_bins = (0, 30), (0,.35), (100, 100)
fig, axs = plt.subplots(1, 2, figsize=(12,6))
_, _, _, mesh = axs[0].hist2d(
    np.array(electrons["PCAL_V"]),
    np.array(electrons["SF"]),
    bins = num_bins,
    range=(low_bin, high_bin),
    norm=mcolors.LogNorm(),
)
axs[0].set_ylabel("SF")
axs[0].set_xlabel("PCAL V (cm)")
axs[0].vlines(14, low_bin[0], low_bin[1], color='red')
divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(mesh, cax=cax)

_, _, _, mesh = axs[1].hist2d(
    np.array(electrons["PCAL_W"]),
    np.array(electrons["SF"]),
    bins = num_bins,
    range=(low_bin, high_bin),
    norm=mcolors.LogNorm(),
)
axs[1].set_ylabel("SF")
axs[1].set_xlabel("PCAL W (cm)")
axs[1].vlines(14, low_bin[0], low_bin[1], color='red')
divider = make_axes_locatable(axs[1])
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(mesh, cax=cax)
plt.tight_layout()
plt.suptitle(plot_title, y=1.02)

In [ ]:
PCAL_fiducial_mask = (electrons["PCAL_W"]>14) & (electrons["PCAL_V"]>14)

In [ ]:
low_bin, high_bin, num_bins = (0, 30), (0,.35), (100, 100)
fig, axs = plt.subplots(1, 2, figsize=(12,6))
_, _, _, mesh = axs[0].hist2d(
    np.array(electrons["PCAL_U"]),
    np.array(electrons["SF"]),
    bins = num_bins,
    range=(low_bin, high_bin),
    norm=mcolors.LogNorm(),
)
axs[0].set_ylabel("SF")
axs[0].set_xlabel("PCAL U (cm)")
divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(mesh, cax=cax)

_, _, _, mesh = axs[1].hist2d(
    np.array(electrons["PCAL_U"][PCAL_fiducial_mask]),
    np.array(electrons["SF"][PCAL_fiducial_mask]),
    bins = num_bins,
    range=(low_bin, high_bin),
    norm=mcolors.LogNorm(),
)
axs[1].set_ylabel("SF")
axs[1].set_xlabel("PCAL U (cm)")
divider = make_axes_locatable(axs[1])
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(mesh, cax=cax)
plt.tight_layout()
plt.suptitle(plot_title, y=1.02)


In [ ]:
region1_low_bin, region1_high_bin, region1_num_bins = (-150, 150), (-150, 150), (250, 250)
region2_low_bin, region2_high_bin, region2_num_bins = (-200, 200), (-200, 200), (250, 250)
region3_low_bin, region3_high_bin, region3_num_bins = (-250, 250), (-250, 250), (250, 250)

fig, axs = plt.subplots(1, 3, figsize=(18,6))
_, _, _, mesh = axs[0].hist2d(
    np.array(electrons["DC_region1_x"]),
    np.array(electrons["DC_region1_y"]),
    bins = region1_num_bins,
    range=(region1_low_bin, region1_high_bin),
    norm=mcolors.LogNorm(),
)
axs[0].set_ylabel("y (cm)")
axs[0].set_xlabel("x (cm)")
axs[0].set_title("DC Region 1")

divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(mesh, cax=cax)

_, _, _, mesh = axs[1].hist2d(
    np.array(electrons["DC_region2_x"]),
    np.array(electrons["DC_region2_y"]),
    bins = region2_num_bins,
    range=(region2_low_bin, region2_high_bin),
    norm=mcolors.LogNorm(),
)
axs[1].set_ylabel("y (cm)")
axs[1].set_xlabel("x (cm)")
axs[1].set_title("DC Region 2")

divider = make_axes_locatable(axs[1])
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(mesh, cax=cax)

_, _, _, mesh = axs[2].hist2d(
    np.array(electrons["DC_region3_x"]),
    np.array(electrons["DC_region3_y"]),
    bins = region3_num_bins,
    range=(region3_low_bin, region3_high_bin),
    norm=mcolors.LogNorm(),
)
axs[2].set_ylabel("y (cm)")
axs[2].set_xlabel("x (cm)")
axs[2].set_title("DC Region 3")

divider = make_axes_locatable(axs[2])
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(mesh, cax=cax)

plt.tight_layout()
plt.suptitle(plot_title, y=1.02)

In [ ]:
distance_to_edge_low_bin = 0
distance_to_edge_high_bin = 20
distance_to_edge_num_bins = 25
bins = np.linspace(distance_to_edge_low_bin, distance_to_edge_high_bin, distance_to_edge_num_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

edge_cut_values = {"region1":4.5, "region2":3.5, "region3":7.5}
fig, axs = plt.subplots(1, 3, figsize=(24, 7))

for region_i in range(3):
    max_value = 0
    for sector in range(num_sectors):
        sector_mask = np.array(electrons["sector"] == sector + 1)
    
        distance_to_edge = np.array(electrons[f"DC_region{region_i+1}_edge"][sector_mask])
        chi2 = np.array(electrons["chi2"][sector_mask])
        ndf = np.array(electrons["NDF"][sector_mask])
        chi2_per_ndf = chi2 / ndf
        bin_indices = np.digitize(distance_to_edge, bins) - 1
    
        bin_means = []
        for i in range(distance_to_edge_num_bins):
            values_in_bin = chi2_per_ndf[bin_indices == i]
            if len(values_in_bin) > 0:
                bin_means.append(np.mean(values_in_bin))
            else:
                bin_means.append(np.nan)
    
        axs[region_i].scatter(bin_centers, bin_means, label=f"Sector {sector + 1}")
        bin_means = np.array(bin_means)[~np.isnan(bin_means)]
        max_bin_means = max(bin_means)
        if max_bin_means>max_value:
            max_value=max_bin_means
    axs[region_i].vlines(edge_cut_values[f"region{region_i+1}"], 0, max_value, color='red')
    axs[region_i].set_title(f"DC Region {region_i+1}")
    axs[region_i].set_xlabel("Distance to Edge (cm)")
    axs[region_i].set_ylabel("Average χ²/NDF")
    axs[region_i].legend(ncols=2, loc='upper right', columnspacing=.8)
    axs[region_i].grid(True)
plt.tight_layout()
plt.suptitle(plot_title, y=1.02)

In [ ]:
DC_cuts = (electrons["DC_region1_edge"]>4.5) & (electrons["DC_region2_edge"]>3.5) & (electrons["DC_region3_edge"]>7.5)

In [ ]:
region1_low_bin, region1_high_bin, region1_num_bins = (-150, 150), (-150, 150), (250, 250)
region2_low_bin, region2_high_bin, region2_num_bins = (-200, 200), (-200, 200), (250, 250)
region3_low_bin, region3_high_bin, region3_num_bins = (-250, 250), (-250, 250), (250, 250)

fig, axs = plt.subplots(1, 3, figsize=(18,6))
_, _, _, mesh = axs[0].hist2d(
    np.array(electrons["DC_region1_x"][DC_cuts]),
    np.array(electrons["DC_region1_y"][DC_cuts]),
    bins = region1_num_bins,
    range=(region1_low_bin, region1_high_bin),
    norm=mcolors.LogNorm(),
)
axs[0].set_ylabel("y (cm)")
axs[0].set_xlabel("x (cm)")
axs[0].set_title("DC Region 1")

divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(mesh, cax=cax)

_, _, _, mesh = axs[1].hist2d(
    np.array(electrons["DC_region2_x"][DC_cuts]),
    np.array(electrons["DC_region2_y"][DC_cuts]),
    bins = region2_num_bins,
    range=(region2_low_bin, region2_high_bin),
    norm=mcolors.LogNorm(),
)
axs[1].set_ylabel("y (cm)")
axs[1].set_xlabel("x (cm)")
axs[1].set_title("DC Region 2")

divider = make_axes_locatable(axs[1])
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(mesh, cax=cax)

_, _, _, mesh = axs[2].hist2d(
    np.array(electrons["DC_region3_x"][DC_cuts]),
    np.array(electrons["DC_region3_y"][DC_cuts]),
    bins = region3_num_bins,
    range=(region3_low_bin, region3_high_bin),
    norm=mcolors.LogNorm(),
)
axs[2].set_ylabel("y (cm)")
axs[2].set_xlabel("x (cm)")
axs[2].set_title("DC Region 3")

divider = make_axes_locatable(axs[2])
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(mesh, cax=cax)

plt.tight_layout()
plt.suptitle(plot_title, y=1.02)

In [ ]:
fiducial_cuts = (PCAL_fiducial_mask) & (DC_cuts)
print(f"Percent of EB electrons remaining after fiducial cut: {round(len(electrons[fiducial_cuts]['pid'])/len(original_event_builder_electrons['pid']), 3)*100}")
print(len(original_event_builder_electrons['pid']))
electrons = electrons[fiducial_cuts]
print(len(electrons["pid"]))

In [ ]:
# plt.hist2d(
#     np.array(electrons["E_PCAL"][track_charge_mask]), 
#     np.array(electrons["E_ECIN"][track_charge_mask]+branches[outer_ecal_name][track_charge_mask]),
#     bins=(500,500),
#     range=[(0,.2), (0,.2)],
#     norm=mcolors.LogNorm());
# plt.xlabel("$E_{PCAL}$ (GeV)")
# plt.ylabel("$E_{ECIN}+E_{ECOUT}$ (GeV)")
# plt.colorbar()
# plt.title(plot_title+"\n Negative tracks")
# plt.axvline(x=0.06, color='red', linewidth=2)
# plt.axvline(x=0.07, color='red', linewidth=2)

# refined_pcal_cut = branches["E_PCAL"][electron_pid_mask]>.07

In [ ]:
# print(f"When we require more than 70 MeV in the PCAL, we keep {round(len(branches['pid'][refined_pcal_cut])/len(branches['pid'][electron_pid_mask]),3)*100}% of EB electrons")

In [ ]:
phi = np.asarray(electrons["phi"]) # in range [-pi, pi]
phi_degrees = (phi) * 180/np.pi
phi_degrees[phi_degrees<0] += 360
electrons["phi_degrees"] = phi_degrees
plt.hist(electrons["phi_degrees"], bins = 360, range=(0,360))
plt.xlabel("$\phi (deg)$")


sector_1_cut = electrons["sector"]==1
sector_2_cut = electrons["sector"]==2
sector_3_cut = electrons["sector"]==3
sector_4_cut = electrons["sector"]==4
sector_5_cut = electrons["sector"]==5
sector_6_cut = electrons["sector"]==6
sector_cuts = [sector_1_cut, sector_2_cut, sector_3_cut, sector_4_cut, sector_5_cut, sector_6_cut]
sector_titles = ["Sector 1", "Sector 2","Sector 3", "Sector 4", "Sector 5", "Sector 6"]

In [ ]:
if make_event_builder_plots:
    low_pcal_bin = .1
    high_pcal_bin = 2
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    fig, axs = plt.subplots(3, 2, figsize=(18, 18))
    axs = axs.flatten()
    sampling_fraction_by_sector = []
    pcal_energy_by_sector = []
    pcal_bins_by_sector = []
    for sector in range(num_sectors):
        overall_mask = (HTCC_mask_nocharge) & (track_charge_mask) & (PCAL_mask_nocharge)
        sector_cut = branches["sector"][overall_mask]==(sector+1)
        total_ecal_energy = np.array(branches["E_PCAL"][overall_mask][sector_cut]+
                                     branches["E_ECIN"][overall_mask][sector_cut]+
                                     branches[outer_ecal_name][overall_mask][sector_cut])
        particle_momentum = np.array(branches["p"][overall_mask][sector_cut])
        pcal_energy = np.array(branches["E_PCAL"][overall_mask][sector_cut])
        sampling_fraction = total_ecal_energy/particle_momentum
        sampling_fraction_by_sector.append(sampling_fraction)
        pcal_energy_by_sector.append(total_ecal_energy)
        hist, pcal_bins, sf_bins, mesh= axs[sector].hist2d(
            total_ecal_energy,
            sampling_fraction,
            bins=(100,100),
            range=[(low_pcal_bin,high_pcal_bin),(0,.4)],
            norm=mcolors.LogNorm())
        pcal_bins_by_sector.append(pcal_bins)
        axs[sector].set_xlabel("$E_{dep}$ (GeV)")
        axs[sector].set_ylabel("$(E_{PCAL}+E_{ECIN}+E_{ECOUTT})/P$")
        axs[sector].set_title(f"Sector {sector+1}") 
        divider = make_axes_locatable(axs[sector])
        cax = divider.append_axes("right", size="5%", pad=0.05)
        
        cbar = fig.colorbar(mesh, cax=cax)
    fig.tight_layout()
    fig.suptitle(plot_title+"\n $E_{PCAL}$>60 MeV and $N_{HTCC PE}$>=2", y=1.05)

In [ ]:
if make_event_builder_plots:
    def gaus(x, a, mu, sigma):
        return a*np.exp(-(x-mu)**2/(2*sigma*sigma))
    def sf_gaussians_by_sector_pcal(sampling_fractions_by_sector,
                                    pcal_by_sector,
                                    pcal_bins_by_sector,
                                    sector_number):
        sf_fit_data = {
            "pcal_low": [],
            "pcal_high": [],
            "pcal_center": [],
            "mu": [],
            "sigma": []
        }
    
        sector_index = sector_number - 1
        fig, axs = plt.subplots(10, 10, figsize=(45, 55))
        fig.subplots_adjust(hspace=0.1, wspace=0.1)
        axs = axs.flatten()
        for i, lower_bin_edge in enumerate(pcal_bins_by_sector[sector_index]):
            if i == len(pcal_bins)-1:
                break
            upper_bin_edge = pcal_bins_by_sector[sector_index][i+1]
            pcal_mask = (pcal_by_sector[sector_index]>lower_bin_edge) & (pcal_by_sector[sector_index]<upper_bin_edge)
            counts, bins, _ = axs[i].hist(sampling_fraction_by_sector[sector_index][pcal_mask], bins=100, range=(0,.4))
            
            bin_centers = (bins[:-1] + bins[1:])/2
            popt, pcov = curve_fit(
                gaus,
                bin_centers,
                counts,
                p0=(len(sampling_fraction_by_sector[sector_index][pcal_mask]),
                    np.mean(sampling_fraction_by_sector[sector_index][pcal_mask]),
                    np.std(sampling_fraction_by_sector[sector_index][pcal_mask]))
            )
            sf_fit_data["pcal_low"].append(lower_bin_edge)
            sf_fit_data["pcal_high"].append(upper_bin_edge)
            sf_fit_data["pcal_center"].append((upper_bin_edge+lower_bin_edge)/2)
            sf_fit_data["mu"].append(popt[1])
            sf_fit_data["sigma"].append(popt[2])
            axs[i].plot(bin_centers, gaus(bin_centers, *popt))
            axs[i].set_xlabel("SF", fontsize=10)
            axs[i].set_title(f"{round(lower_bin_edge,3)} GeV < $E_{{dep}}$ < {round(upper_bin_edge,3)} GeV", fontsize=10)
            axs[i].tick_params(axis='both', which='major', labelsize=10)
        fig.suptitle(f"Sector {sector_number}", y = 1.01)
        fig.tight_layout()
        sf_fit_data_df = pd.DataFrame(sf_fit_data)
        return sf_fit_data_df

In [ ]:
if make_event_builder_plots:
    sf_fit_pcal_df_by_sector = []
    for sector in range(num_sectors):
        sector_number = sector + 1
        sf_df = sf_gaussians_by_sector_pcal(sampling_fraction_by_sector,
                                       pcal_energy_by_sector,
                                       pcal_bins_by_sector,
                                       sector_number)
        sf_fit_pcal_df_by_sector.append(sf_df)

In [ ]:
if make_event_builder_plots:
    def sf_fit_function_pcal(x, a, b, c):
        return a + b/x + c/(x*x)

In [ ]:
if make_event_builder_plots:
    popt_mu_by_sector, popt_sigma_by_sector = [], []
    for sector in range(num_sectors):
    
        fig, axs = plt.subplots(1, 2, figsize=(15, 6))
        fig.subplots_adjust(hspace=0.1, wspace=0.3)
        sf_fit_data_df = sf_fit_pcal_df_by_sector[sector]
        axs[0].scatter(sf_fit_data_df["pcal_center"].tolist(), sf_fit_data_df["mu"].tolist())
        axs[0].set_xlabel("$E_{dep}$ bin center (GeV)")
        axs[0].set_ylabel("SF $\mu$")
        popt_mu, pcov_mu = curve_fit(sf_fit_function_pcal, sf_fit_data_df["pcal_center"].tolist(), sf_fit_data_df["mu"].tolist(), p0=(.2, .001, .00001))
        axs[0].plot(sf_fit_data_df["pcal_center"].tolist(), sf_fit_function_pcal(np.array(sf_fit_data_df["pcal_center"].tolist()), *popt_mu), color='red')
        print(popt_mu)
        axs[1].scatter(sf_fit_data_df["pcal_center"].tolist(), sf_fit_data_df["sigma"].tolist())
        axs[1].set_xlabel("$E_{dep}$ bin center (GeV)")
        axs[1].set_ylabel("SF $\sigma$")
        popt_sigma, pcov_sigma = curve_fit(sf_fit_function_pcal, sf_fit_data_df["pcal_center"].tolist(), sf_fit_data_df["sigma"].tolist(), p0=(.002, .001, .00001))
        axs[1].plot(sf_fit_data_df["pcal_center"].tolist(), sf_fit_function_pcal(np.array(sf_fit_data_df["pcal_center"].tolist()), *popt_sigma), color='red')
        fig.suptitle(f"{plot_title}\nSector {sector+1}", y=1.03) 
        popt_mu_by_sector.append(popt_mu)
        popt_sigma_by_sector.append(popt_sigma)

In [ ]:
if make_event_builder_plots:
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    fig, axs = plt.subplots(3, 2, figsize=(18, 18))
    axs = axs.flatten()
    sampling_fraction_by_sector = []
    pcal_energy_by_sector = []
    pcal_bins_by_sector = []
    for sector in range(num_sectors):
        overall_mask = (HTCC_mask_nocharge) & (track_charge_mask) & (PCAL_mask_nocharge)
        sector_cut = branches["sector"][overall_mask]==(sector+1)
        total_ecal_energy = np.array(branches["E_PCAL"][overall_mask][sector_cut]+
                                     branches["E_ECIN"][overall_mask][sector_cut]+
                                     branches[outer_ecal_name][overall_mask][sector_cut])
        particle_momentum = np.array(branches["p"][overall_mask][sector_cut])
        pcal_energy = np.array(branches["E_PCAL"][overall_mask][sector_cut])
        sampling_fraction = total_ecal_energy/particle_momentum
        sampling_fraction_by_sector.append(sampling_fraction)
        pcal_energy_by_sector.append(pcal_energy)
        
        sf_fit_data_df = sf_fit_pcal_df_by_sector[sector]
        fit_mu = sf_fit_function_pcal(pcal_energy, *popt_mu)
        fit_sigma = sf_fit_function_pcal(pcal_energy, *popt_sigma)
        valid_mask = (sampling_fraction < (fit_mu + 5*fit_sigma)) & (sampling_fraction > (fit_mu - 5*fit_sigma))
        
        hist, pcal_bins, sf_bins, mesh= axs[sector].hist2d(
            total_ecal_energy,
            sampling_fraction,
            bins=(100,100),
            range=[(low_pcal_bin,high_pcal_bin),(0,.4)],
            norm=mcolors.LogNorm())
        pcal_bins_by_sector.append(pcal_bins)
        axs[sector].set_xlabel("$E_{dep}$ (GeV)")
        axs[sector].set_ylabel("$(E_{PCAL}+E_{ECIN}+E_{ECOUTT})/P$")
        axs[sector].set_title(f"Sector {sector+1}") 
        divider = make_axes_locatable(axs[sector])
        cax = divider.append_axes("right", size="5%", pad=0.05)
        
        cbar = fig.colorbar(mesh, cax=cax)
    
    
        axs[sector].plot(sf_fit_data_df["pcal_center"].tolist(), sf_fit_function_pcal(np.array(sf_fit_data_df["pcal_center"].tolist()), *popt_mu), color='black')
        axs[sector].plot(
            sf_fit_data_df["pcal_center"].tolist(),
            sf_fit_function_pcal(np.array(sf_fit_data_df["pcal_center"].tolist()), *popt_mu) + 5*sf_fit_function_pcal(np.array(sf_fit_data_df["pcal_center"].tolist()), *popt_sigma),
            color='red'
        )
        axs[sector].plot(
            sf_fit_data_df["pcal_center"].tolist(),
            sf_fit_function_pcal(np.array(sf_fit_data_df["pcal_center"].tolist()), *popt_mu) - 5*sf_fit_function_pcal(np.array(sf_fit_data_df["pcal_center"].tolist()), *popt_sigma),
            color='red'
        )
    
        print(f"Sector {sector+1} % remaining events = {len(particle_momentum[valid_mask])/len(particle_momentum)}")
    fig.tight_layout()
    fig.suptitle(plot_title+"\n $E_{PCAL}$>60 MeV and $N_{HTCC PE}$>=2", y=1.05)

In [ ]:
if make_event_builder_plots:
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    fig, axs = plt.subplots(3, 2, figsize=(18, 18))
    axs = axs.flatten()
    sampling_fraction_by_sector = []
    pcal_energy_by_sector = []
    pcal_bins_by_sector = []
    for sector in range(num_sectors):
        overall_mask = branches["pid"]==11
        sector_cut = branches["sector"][overall_mask]==(sector+1)
        total_ecal_energy = np.array(branches["E_PCAL"][overall_mask][sector_cut]+
                                     branches["E_ECIN"][overall_mask][sector_cut]+
                                     branches[outer_ecal_name][overall_mask][sector_cut])
        particle_momentum = np.array(branches["p"][overall_mask][sector_cut])
        pcal_energy = np.array(branches["E_PCAL"][overall_mask][sector_cut])
        sampling_fraction = total_ecal_energy/particle_momentum
        sampling_fraction_by_sector.append(sampling_fraction)
        pcal_energy_by_sector.append(pcal_energy)
        
        sf_fit_data_df = sf_fit_pcal_df_by_sector[sector]
        fit_mu = sf_fit_function_pcal(pcal_energy, *popt_mu)
        fit_sigma = sf_fit_function_pcal(pcal_energy, *popt_sigma)
        valid_mask = (sampling_fraction < (fit_mu + 5*fit_sigma)) & (sampling_fraction > (fit_mu - 5*fit_sigma))
        
        hist, pcal_bins, sf_bins, mesh= axs[sector].hist2d(
            total_ecal_energy,
            sampling_fraction,
            bins=(100,100),
            range=[(low_pcal_bin,high_pcal_bin),(0,.4)],
            norm=mcolors.LogNorm())
        pcal_bins_by_sector.append(pcal_bins)
        axs[sector].set_xlabel("$E_{dep}$ (GeV)")
        axs[sector].set_ylabel("$(E_{PCAL}+E_{ECIN}+E_{ECOUTT})/P$")
        axs[sector].set_title(f"Sector {sector+1}") 
        divider = make_axes_locatable(axs[sector])
        cax = divider.append_axes("right", size="5%", pad=0.05)
        
        cbar = fig.colorbar(mesh, cax=cax)
    
    
        axs[sector].plot(sf_fit_data_df["pcal_center"].tolist(), sf_fit_function_pcal(np.array(sf_fit_data_df["pcal_center"].tolist()), *popt_mu), color='black')
        axs[sector].plot(
            sf_fit_data_df["pcal_center"].tolist(),
            sf_fit_function_pcal(np.array(sf_fit_data_df["pcal_center"].tolist()), *popt_mu) + 5*sf_fit_function_pcal(np.array(sf_fit_data_df["pcal_center"].tolist()), *popt_sigma),
            color='red'
        )
        axs[sector].plot(
            sf_fit_data_df["pcal_center"].tolist(),
            sf_fit_function_pcal(np.array(sf_fit_data_df["pcal_center"].tolist()), *popt_mu) - 5*sf_fit_function_pcal(np.array(sf_fit_data_df["pcal_center"].tolist()), *popt_sigma),
            color='red'
        )
    
        print(f"Sector {sector+1} % remaining events = {len(particle_momentum[valid_mask])/len(particle_momentum)}")
    fig.tight_layout()
    fig.suptitle(plot_title+"\n pid==11", y=1.02)

In [ ]:
def ecin_epcal_cut(ecin):
    if is_simulation:
        return (-.22/.18)*ecin + .22
    else:
        return (-.22/.15)*ecin + .22

In [ ]:
ecal_energy = np.array(electrons["E_ECIN"]/electrons["p"])
hist, ecin_bins, epcal_bins, mesh = plt.hist2d(
    ecal_energy, 
    np.array(electrons["E_PCAL"]/electrons["p"]),
    bins=(250,250),
    range=[(0,.2), (0,.25)],
    norm=mcolors.LogNorm());
plt.ylabel("$E_{PCAL}$/p")
plt.xlabel("$E_{ECIN}$/p")
plt.colorbar()
plt.title(plot_title)
plt.plot(ecin_bins.tolist(), ecin_epcal_cut(np.array(ecin_bins)), color='red')
ecal_diagonal_mask = np.array(electrons["E_PCAL"]/electrons["p"]) > ecin_epcal_cut(ecal_energy)
ecal_diagonal_mask[electrons["p"] < 4.5] = True

In [ ]:
momentum_bins = np.arange(4.5, 11.5, 1)
fig, axs = plt.subplots(3, 2, figsize=(12, 18))
axs = axs.flatten()
for i, p_bin in enumerate(momentum_bins):
    if i == len(momentum_bins)-1:
        break
    momentum_mask = (electrons["p"] > p_bin) & (electrons["p"] < momentum_bins[i+1])
    if len(np.array(electrons["E_ECIN"]/electrons["p"])[momentum_mask])==0:
        continue
    hist, ecin_bins, epcal_bins, mesh = axs[i].hist2d(
        np.array(electrons["E_ECIN"]/electrons["p"])[momentum_mask], 
        np.array(electrons["E_PCAL"]/electrons["p"])[momentum_mask],
        bins=(100,100),
        range=[(0,.2), (0,.25)],
        norm=mcolors.LogNorm());
    axs[i].set_ylabel("$E_{PCAL}$/p")
    axs[i].set_xlabel("$E_{ECIN}$/p")
    axs[i].set_title(f"{p_bin} < p < {momentum_bins[i+1]}")
    axs[i].plot(ecin_bins.tolist(), ecin_epcal_cut(np.array(ecin_bins)), color='red')
    divider = make_axes_locatable(axs[i])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(mesh, cax=cax)
plt.tight_layout()

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
fig, axs = plt.subplots(3, 2, figsize=(18, 18))
axs = axs.flatten()
for sector in range(num_sectors):
    sector_cut = electrons["sector"]==(sector+1)
    if len(np.array(electrons["E_ECIN"]/electrons["p"])[sector_cut])==0:
        continue
    hist, ecin_bins, epcal_bins, mesh = axs[sector].hist2d(
        np.array(electrons["E_ECIN"]/electrons["p"])[sector_cut], 
        np.array(electrons["E_PCAL"]/electrons["p"])[sector_cut],
        bins=(100,100),
        range=[(0,.2), (0,.25)],
        norm=mcolors.LogNorm())
    axs[sector].set_xlabel("$E_{ECIN}$/p")
    axs[sector].set_ylabel("$E_{PCAL}$/p")
    axs[sector].set_title(f"Sector {sector+1}") 
    divider = make_axes_locatable(axs[sector])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(mesh, cax=cax)
    axs[sector].plot(ecin_bins.tolist(), ecin_epcal_cut(np.array(ecin_bins)), color='red')
fig.tight_layout()
fig.suptitle(plot_title, y=1.)

In [ ]:
print(f"Percent of EB electrons remaining after fiducial cut: {round(len(electrons[ecal_diagonal_mask]['pid'])/len(original_event_builder_electrons['pid']), 3)*100}")
electrons = electrons[ecal_diagonal_mask]

In [ ]:
hist, ecin_bins, epcal_bins, mesh = plt.hist2d(
    np.array(electrons["E_ECIN"]/electrons["p"]), 
    np.array(electrons["E_PCAL"]/electrons["p"]),
    bins=(250, 250),
    range=[(0,.2), (0,.25)],
    norm=mcolors.LogNorm());
plt.ylabel("$E_{PCAL}$/p")
plt.xlabel("$E_{ECIN}$/p")
plt.colorbar()
plt.title(plot_title)
plt.plot(ecin_bins.tolist(), ecin_epcal_cut(np.array(ecin_bins)), color='red')

In [ ]:
def sf_fit_function(x, a, b, c):
    return a + b/x + c/(x*x)

In [ ]:
low_edep_bin = .6
high_edep_bin = 1.6
low_sf_bin = .1
high_sf_bin = .35

### Doing SF vs Edep cuts

In [ ]:
electrons["total_ecal_energy"] = electrons["E_PCAL"] + electrons["E_ECIN"] + electrons["E_ECOUT"]

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
fig, axs = plt.subplots(3, 2, figsize=(18, 18))
axs = axs.flatten()
sampling_fraction_by_sector = []
edep_by_sector = []
edep_bins_by_sector = []
for sector in range(num_sectors):
    sector_cut = electrons["sector"]==(sector+1)
    sector_cut = (sector_cut)
    total_ecal_energy = np.array(electrons["total_ecal_energy"][sector_cut])
    sampling_fraction = np.array(electrons["SF"][sector_cut])
    sampling_fraction_by_sector.append(sampling_fraction)
    edep_by_sector.append(total_ecal_energy)
    
    hist, edep_bins, sf_bins, mesh= axs[sector].hist2d(
        total_ecal_energy,
        sampling_fraction,
        bins=(100,100),
        range=[(low_edep_bin, high_edep_bin),(low_sf_bin, high_sf_bin)],
        norm=mcolors.LogNorm())
    edep_bins_by_sector.append(edep_bins)
    axs[sector].set_xlabel("$E_{dep}$ (GeV)")
    axs[sector].set_ylabel("$(E_{PCAL}+E_{ECIN}+E_{ECOUT})/P$")
    axs[sector].set_title(f"Sector {sector+1}") 
    divider = make_axes_locatable(axs[sector])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(mesh, cax=cax)
fig.tight_layout()
fig.suptitle(plot_title, y=1.)

### 

In [ ]:
def gaus(x, a, mu, sigma):
    return a*np.exp(-(x-mu)**2/(2*sigma*sigma))

In [ ]:
def sf_gaussians_by_sector(sampling_fractions_by_sector,
                           xaxis_by_sector,
                           xaxis_bins_by_sector,
                           sector_number,
                           xaxis_name):
    sf_fit_data = {
        "bin_low": [],
        "bin_high": [],
        "bin_center": [],
        "mu": [],
        "sigma": []
    }
    sector_index = sector_number - 1
    fig, axs = plt.subplots(10, 10, figsize=(45, 55))
    fig.subplots_adjust(hspace=0.1, wspace=0.1)
    axs = axs.flatten()
    for i, lower_bin_edge in enumerate(xaxis_bins_by_sector[sector_index]):
        if i == len(xaxis_bins_by_sector[sector_index])-1:
            break
        upper_bin_edge = xaxis_bins_by_sector[sector_index][i+1]
        xaxis_bin_mask = (xaxis_by_sector[sector_index]>lower_bin_edge) & (xaxis_by_sector[sector_index]<upper_bin_edge)
        sector_sf_mask = (sampling_fraction_by_sector[sector_index][xaxis_bin_mask] > .2) & (sampling_fraction_by_sector[sector_index][xaxis_bin_mask] <.27)
        
        counts, bins, _ = axs[i].hist(sampling_fraction_by_sector[sector_index][xaxis_bin_mask], bins=100, range=(low_sf_bin, high_sf_bin))
        
        bin_centers = (bins[:-1] + bins[1:])/2
        sf_mask = (bin_centers>.2) & (bin_centers<.3)
        popt, pcov = curve_fit(
            gaus,
            bin_centers[sf_mask],
            counts[sf_mask],
            p0=(len(sampling_fraction_by_sector[sector_index][xaxis_bin_mask][sector_sf_mask]),
                np.mean(sampling_fraction_by_sector[sector_index][xaxis_bin_mask][sector_sf_mask]),
                np.std(sampling_fraction_by_sector[sector_index][xaxis_bin_mask][sector_sf_mask]))
        )
        sf_fit_data["bin_low"].append(lower_bin_edge)
        sf_fit_data["bin_high"].append(upper_bin_edge)
        sf_fit_data["bin_center"].append((upper_bin_edge+lower_bin_edge)/2)
        sf_fit_data["mu"].append(popt[1])
        sf_fit_data["sigma"].append(popt[2])
        axs[i].plot(bin_centers, gaus(bin_centers, *popt))
        axs[i].set_xlabel("SF", fontsize=10)
        axs[i].set_title(f"{round(lower_bin_edge,3)} GeV < {xaxis_name} < {round(upper_bin_edge,3)} GeV", fontsize=10)
        axs[i].tick_params(axis='both', which='major', labelsize=10)
    fig.suptitle(f"Sector {sector_number}", y = 1.01)
    fig.tight_layout()
    sf_fit_data_df = pd.DataFrame(sf_fit_data)
    return sf_fit_data_df

In [ ]:
sf_fit_data_df_by_sector = []
for sector in range(num_sectors):
    sector_number = sector + 1
    sf_df = sf_gaussians_by_sector(sampling_fraction_by_sector,
                                   edep_by_sector,
                                   edep_bins_by_sector,
                                   sector_number,
                                   xaxis_name="$E_{dep}$")
    sf_fit_data_df_by_sector.append(sf_df)

In [ ]:
def sf_fit_function(x, a, b, c, d):
    return a + b*x  + c*(x*x) + d*(x*x*x)
# def sf_fit_function(x, a, b, c):
    # return a + b/x + c/(x*x)
def quad_fit_function(x,a,b,c):
    return a*x*x+b*x+c
def mean_fit_function(x, a, b, c):
    return quad_fit_function(x, a, b, c)

In [ ]:
popt_mu_by_sector, popt_sigma_by_sector = [], []
for sector in range(num_sectors):

    fig, axs = plt.subplots(1, 2, figsize=(15, 6))
    fig.subplots_adjust(hspace=0.1, wspace=0.3)
    sf_fit_data_df = sf_fit_data_df_by_sector[sector]
    axs[0].scatter(sf_fit_data_df["bin_center"].tolist(), sf_fit_data_df["mu"].tolist())
    axs[0].set_xlabel("bin center (GeV)")
    axs[0].set_ylabel("SF $\mu$")
    popt_mu, pcov_mu = curve_fit(sf_fit_function, sf_fit_data_df["bin_center"].tolist(), sf_fit_data_df["mu"].tolist(), p0=(.2, .001, .00001, .00001))
    axs[0].plot(sf_fit_data_df["bin_center"].tolist(), sf_fit_function(np.array(sf_fit_data_df["bin_center"].tolist()), *popt_mu), color='red')
    print(popt_mu)
    axs[1].scatter(sf_fit_data_df["bin_center"].tolist(), sf_fit_data_df["sigma"].tolist())
    axs[1].set_xlabel("bin center (GeV)")
    axs[1].set_ylabel("SF $\sigma$")
    popt_sigma, pcov_sigma = curve_fit(sf_fit_function, sf_fit_data_df["bin_center"].tolist(), sf_fit_data_df["sigma"].tolist(), p0=(.002, .001, .00001, .00001))
    axs[1].plot(sf_fit_data_df["bin_center"].tolist(), sf_fit_function(np.array(sf_fit_data_df["bin_center"].tolist()), *popt_sigma), color='red')
    fig.suptitle(f"{plot_title}\nSector {sector+1}", y=1.03) 
    popt_mu_by_sector.append(popt_mu)
    popt_sigma_by_sector.append(popt_sigma)

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(18, 18))
axs = axs.flatten()

valid_mask_by_sector = []
for sector in range(num_sectors):
    popt_mu = popt_mu_by_sector[sector]
    popt_sigma = popt_sigma_by_sector[sector]
    edep = edep_by_sector[sector]
    sampling_fraction = sampling_fraction_by_sector[sector]
    sf_fit_data_df = sf_fit_data_df_by_sector[sector]
    
    fit_mu = sf_fit_function(edep, *popt_mu)
    fit_sigma = sf_fit_function(edep, *popt_sigma)
    valid_mask = (sampling_fraction < (fit_mu + 3.5*fit_sigma)) & (sampling_fraction > (fit_mu - 3.5*fit_sigma))
    valid_mask_by_sector.append(valid_mask)
    
    hist, edep_bins, sf_bins, mesh= axs[sector].hist2d(
        edep,
        sampling_fraction,
        bins=(100,100),
        range=[(0.3,2.1),(.15,.32)],
        norm=mcolors.LogNorm()
    )

    edep_bin_centers = (edep_bins[:-1] + edep_bins[1:])/2
    
    axs[sector].set_xlabel("$E_{dep}$ (GeV)")
    axs[sector].set_ylabel("$(E_{PCAL}+E_{ECIN}+E_{ECOUTT})/P$")
    axs[sector].set_title(f"Sector {sector+1}") 
    divider = make_axes_locatable(axs[sector])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(mesh, cax=cax)

    axs[sector].plot(edep_bin_centers.tolist(), sf_fit_function(np.array(edep_bin_centers.tolist()), *popt_mu), color='black')
    axs[sector].plot(
        edep_bin_centers.tolist(),
        sf_fit_function(np.array(edep_bin_centers.tolist()), *popt_mu) + 3.5*sf_fit_function(np.array(edep_bin_centers.tolist()), *popt_sigma),
        color='red'
    )
    axs[sector].plot(
        edep_bin_centers.tolist(),
        sf_fit_function(np.array(edep_bin_centers.tolist()), *popt_mu) - 3.5*sf_fit_function(np.array(edep_bin_centers.tolist()), *popt_sigma),
        color='red'
    )

    print(f"Sector {sector+1} % remaining events = {len(edep[valid_mask])/len(edep)}")
fig.tight_layout()
fig.suptitle(plot_title, y=1.)

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(18, 18))
axs = axs.flatten()
sf_mask_by_sector = []
for sector in range(num_sectors):
    popt_mu = popt_mu_by_sector[sector]
    popt_sigma = popt_sigma_by_sector[sector]
    edep = edep_by_sector[sector]
    sampling_fraction = sampling_fraction_by_sector[sector]
    sf_fit_data_df = sf_fit_data_df_by_sector[sector]
    
    fit_mu = sf_fit_function(edep, *popt_mu)
    fit_sigma = sf_fit_function(edep, *popt_sigma)
    sf_mask = (sampling_fraction < (fit_mu + 3.5*fit_sigma)) & (sampling_fraction > (fit_mu - 3.5*fit_sigma))
    
    sf_mask_by_sector.append(sf_mask)
    
    hist, p_bins, sf_bins, mesh= axs[sector].hist2d(
        edep[sf_mask],
        sampling_fraction[sf_mask],
        bins=(100,100),
        range=[(low_edep_bin,2.5),(.12,.32)],
        norm=mcolors.LogNorm()
    )
    axs[sector].set_xlabel("$E_{dep}$ (GeV)")
    axs[sector].set_ylabel("$(E_{PCAL}+E_{ECIN}+E_{ECOUTT})/P$")
    axs[sector].set_title(f"Sector {sector+1}") 
    divider = make_axes_locatable(axs[sector])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(mesh, cax=cax)


    axs[sector].plot(sf_fit_data_df["bin_center"].tolist(), sf_fit_function(np.array(sf_fit_data_df["bin_center"].tolist()), *popt_mu), color='black')
    axs[sector].plot(
        sf_fit_data_df["bin_center"].tolist(),
        sf_fit_function(np.array(sf_fit_data_df["bin_center"].tolist()), *popt_mu) + 3.5*sf_fit_function(np.array(sf_fit_data_df["bin_center"].tolist()), *popt_sigma),
        color='red'
    )
    axs[sector].plot(
        sf_fit_data_df["bin_center"].tolist(),
        sf_fit_function(np.array(sf_fit_data_df["bin_center"].tolist()), *popt_mu) - 3.5*sf_fit_function(np.array(sf_fit_data_df["bin_center"].tolist()), *popt_sigma),
        color='red'
    )
    print(f"Sector {sector+1} % remaining events = {len(edep[sf_mask])/len(edep)}")
fig.tight_layout()
fig.suptitle(plot_title+"\n After SF fit cut", y=1.02)

In [ ]:
# combined_mask = (sf_mask_all_sectors)
# hist, ecin_bins, epcal_bins, mesh = plt.hist2d(
#     ecal_energy[combined_mask], 
#     np.array(branches["E_PCAL"][electron_pid_mask][ecal_diagonal_mask][combined_mask]/branches["p"][electron_pid_mask][ecal_diagonal_mask][combined_mask]),
#     bins=(100,100),
#     range=[(0,.25), (0,.34)],
#     norm=mcolors.LogNorm());
# plt.ylabel("$E_{PCAL}$/p")
# plt.xlabel("$E_{ECIN}$/p")
# plt.colorbar()
# plt.title(plot_title+"\n EB electrons\n with diagonal and SF cuts")
# plt.plot(ecin_bins.tolist(), ecin_epcal_cut(np.array(ecin_bins)), color='red')
# print(len(branches["p"][electron_pid_mask][combined_mask])/len(branches["p"][electron_pid_mask]))

In [ ]:
# fig, axs = plt.subplots(3, 2, figsize=(18, 18))
# axs = axs.flatten()
# for sector in range(num_sectors):
#     popt_mu = popt_mu_by_sector[sector]
#     popt_sigma = popt_sigma_by_sector[sector]
#     edep = edep_by_sector[sector]
#     sampling_fraction = sampling_fraction_by_sector[sector]
#     sf_fit_data_df = sf_fit_data_df_by_sector[sector]
#     ecal_diagonal_mask_sector = ecal_diagonal_mask[branches["sector"][electron_pid_mask]==sector+1]
#     fit_mu = sf_fit_function(edep, *popt_mu)
#     fit_sigma = sf_fit_function(edep, *popt_sigma)
    
#     valid_mask = (sampling_fraction < (fit_mu + 3.5*fit_sigma)) & (sampling_fraction > (fit_mu - 3.5*fit_sigma)) & (ecal_diagonal_mask_sector)
    
                
#     hist, p_bins, sf_bins, mesh= axs[sector].hist2d(
#         edep[valid_mask],
#         sampling_fraction[valid_mask],
#         bins=(100,100),
#         range=[(low_edep_bin,high_edep_bin),(.12,.32)],
#         norm=mcolors.LogNorm()
#     )
#     axs[sector].set_xlabel("$E_{dep} (GeV)$")
#     axs[sector].set_ylabel("$(E_{PCAL}+E_{ECIN}+E_{ECOUTT})/P$")
#     axs[sector].set_title(f"Sector {sector+1}") 
#     divider = make_axes_locatable(axs[sector])
#     cax = divider.append_axes("right", size="5%", pad=0.05)
#     cbar = fig.colorbar(mesh, cax=cax)


#     axs[sector].plot(sf_fit_data_df["bin_center"].tolist(), sf_fit_function(np.array(sf_fit_data_df["bin_center"].tolist()), *popt_mu), color='black')
#     axs[sector].plot(
#         sf_fit_data_df["bin_center"].tolist(),
#         sf_fit_function(np.array(sf_fit_data_df["bin_center"].tolist()), *popt_mu) + 3.5*sf_fit_function(np.array(sf_fit_data_df["bin_center"].tolist()), *popt_sigma),
#         color='red'
#     )
#     axs[sector].plot(
#         sf_fit_data_df["bin_center"].tolist(),
#         sf_fit_function(np.array(sf_fit_data_df["bin_center"].tolist()), *popt_mu) - 3.5*sf_fit_function(np.array(sf_fit_data_df["bin_center"].tolist()), *popt_sigma),
#         color='red'
#     )
#     print(f"Sector {sector+1} % remaining events = {len(edep[valid_mask])/len(edep)}")
# fig.tight_layout()
# fig.suptitle(plot_title+"\n After partial SF cut", y=1.02)

In [ ]:
# electron_candidate_mask =  (refined_pcal_cut[ecal_diagonal_mask]) & (electron_status_mask[ecal_diagonal_mask]) & (sf_mask_all_sectors)
# print(len(branches["sector"][electron_pid_mask][ecal_diagonal_mask][electron_candidate_mask])/len(branches["sector"][electron_pid_mask]))

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(18, 18))
axs = axs.flatten()
for sector in range(num_sectors):
    sector_cut = event_builder_electrons["sector"]==(sector+1)
    
    vector_x = np.array(event_builder_electrons["v_x"][sector_cut])
    axs[sector].hist(
        vector_x,
        bins=100,
        range=(-2,2),
        histtype='step'
    )
    # p_bins_by_sector.append(p_bins)
    axs[sector].set_xlabel("$v_{x}$ (cm)")
    axs[sector].set_title(f"Sector {sector+1}") 
    # divider = make_axes_locatable(axs[sector])
    # cax = divider.append_axes("right", size="5%", pad=0.05)
    
    # cbar = fig.colorbar(mesh, cax=cax)
fig.tight_layout()
fig.suptitle(plot_title+"\n Event Builder electrons", y=1.02)

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(18, 18))
axs = axs.flatten()
for sector in range(num_sectors):
    sector_cut = electrons["sector"]==(sector+1)
    
    vector_x = np.array(electrons["v_x"][sector_cut][sf_mask_by_sector[sector]])
    axs[sector].hist(
        vector_x,
        bins=100,
        range=(-2,2),
        histtype='step'
    )
    # p_bins_by_sector.append(p_bins)
    axs[sector].set_xlabel("$v_{x}$ (cm)")
    axs[sector].set_title(f"Sector {sector+1}") 
    # divider = make_axes_locatable(axs[sector])
    # cax = divider.append_axes("right", size="5%", pad=0.05)
    
    # cbar = fig.colorbar(mesh, cax=cax)
fig.tight_layout()
fig.suptitle(plot_title+"\n Electron candidates", y=1.02)

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(18, 18))
axs = axs.flatten()
for sector in range(num_sectors):
    sector_cut = electrons["sector"]==(sector+1)
    
    vector_z = np.array(electrons["v_y"][sector_cut][sf_mask_by_sector[sector]])
    axs[sector].hist(
        vector_z,
        bins=100,
        range=(-2,2),
        histtype='step'
    )
    # p_bins_by_sector.append(p_bins)
    axs[sector].set_xlabel("$v_{y}$ (cm)")
    axs[sector].set_title(f"Sector {sector+1}") 
    # divider = make_axes_locatable(axs[sector])
    # cax = divider.append_axes("right", size="5%", pad=0.05)
    
    # cbar = fig.colorbar(mesh, cax=cax)
fig.tight_layout()
fig.suptitle(plot_title+"\n Electron candidates", y=1.02)

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(18, 18))
axs = axs.flatten()
for sector in range(num_sectors):
    sector_cut = event_builder_electrons["sector"]==(sector+1)
    
    vector_z = np.array(event_builder_electrons["v_y"][sector_cut])
    axs[sector].hist(
        vector_z,
        bins=100,
        range=(-5,5),
        histtype='step'
    )
    # p_bins_by_sector.append(p_bins)
    axs[sector].set_xlabel("$v_{y}$ (cm)")
    axs[sector].set_title(f"Sector {sector+1}") 
    # divider = make_axes_locatable(axs[sector])
    # cax = divider.append_axes("right", size="5%", pad=0.05)
    
    # cbar = fig.colorbar(mesh, cax=cax)
fig.tight_layout()
fig.suptitle(plot_title+"\n Event Builder electrons", y=1.02)

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(18, 18))
axs = axs.flatten()
for sector in range(num_sectors):
    sector_cut = event_builder_electrons["sector"]==(sector+1)
    
    vector_z = np.array(event_builder_electrons["v_z"][sector_cut])
    axs[sector].hist(
        vector_z,
        bins=100,
        range=(-20,20),
        histtype='step'
    )
    # p_bins_by_sector.append(p_bins)
    axs[sector].set_xlabel("$v_{z}$ (cm)")
    axs[sector].set_title(f"Sector {sector+1}") 
    # divider = make_axes_locatable(axs[sector])
    # cax = divider.append_axes("right", size="5%", pad=0.05)
    axs[sector].set_ylim(0, 150000)
    # cbar = fig.colorbar(mesh, cax=cax)
fig.tight_layout()
fig.suptitle(plot_title+"\n Event Builder electrons", y=1.02)

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(18, 18))
axs = axs.flatten()
for sector in range(num_sectors):
    sector_cut = electrons["sector"]==(sector+1)
    
    vector_z = np.array(electrons["v_z"][sector_cut])
    axs[sector].hist(
        vector_z,
        bins=100,
        range=(-20,20),
        histtype='step'
    )
    # p_bins_by_sector.append(p_bins)
    axs[sector].set_xlabel("$v_{z}$ (cm)")
    axs[sector].set_title(f"Sector {sector+1}") 
    # divider = make_axes_locatable(axs[sector])
    # cax = divider.append_axes("right", size="5%", pad=0.05)
    # axs[sector].set_ylim(0, 150000)
    # cbar = fig.colorbar(mesh, cax=cax)
fig.tight_layout()
fig.suptitle(plot_title+"\n Electron candidates", y=1.02)

In [ ]:
fig=plt.figure(figsize=(18,18))
for sector in range(num_sectors):
    sector_cut = event_builder_electrons["sector"]==(sector+1)
    vector_z = np.array(event_builder_electrons["v_z"][sector_cut])
    plt.hist(
        vector_z,
        bins=100,
        range=(-12,5),
        histtype='step',
        label=f"Sector {sector+1}",
        density=True
    )
    # p_bins_by_sector.append(p_bins)
    plt.xlabel("$v_{z}$ (cm)")
    # divider = make_axes_locatable(axs[sector])
    # cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.legend()
    
    # cbar = fig.colorbar(mesh, cax=cax)
fig.tight_layout()
fig.suptitle(plot_title+"\n EB electrons", y=1.04)

In [ ]:
fig=plt.figure(figsize=(18,18))
for sector in range(num_sectors):
    sector_cut = electrons["sector"]==(sector+1)
    vector_z = np.array(electrons["v_z"][sector_cut][sf_mask_by_sector[sector]])
    plt.hist(
        vector_z,
        bins=100,
        range=(-12,5),
        histtype='step',
        label=f"Sector {sector+1}",
        density=True
    )
    # p_bins_by_sector.append(p_bins)
    plt.xlabel("$v_{z}$ (cm)")
    # divider = make_axes_locatable(axs[sector])
    # cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.legend()
    
    # cbar = fig.colorbar(mesh, cax=cax)
fig.tight_layout()
fig.suptitle(plot_title+"\n Electron candidates", y=1.04)

In [ ]:
def double_gaussian(x, amp1, mean1, sigma1, amp2, mean2, sigma2):
    return amp1 * np.exp( -(x - mean1)**2 / (2*sigma1) ) + \
           amp2 * np.exp( -(x - mean2)**2 / (2*sigma2) )

In [ ]:
if target_vertex_selection:
    fig, axs = plt.subplots(3, 2, figsize=(18, 18))
    axs = axs.flatten()
    z_fit_parameters_all_sectors = []
    for sector in range(num_sectors):
        sector_cut = electrons["sector"]==(sector+1)
        
        vertex_z = np.array(electrons["v_z"][sector_cut][sf_mask_by_sector[sector]])
        vertex_z_counts, vertex_z_bins, _ = axs[sector].hist(
            vertex_z,
            bins=100,
            range=(-12,5),
            histtype='step'
        )
        vertex_z_bin_centers = (vertex_z_bins[:-1] + vertex_z_bins[1:])/2
        lower_fit_range = -12
        upper_fit_range = 12
    
        amplitude_guess = np.max(vertex_z_counts)
        mean_z_guess = ak.mean(vertex_z)
        std_z_guess = ak.std(vertex_z)
        
        print(amplitude_guess)
        z_fit_parameters, z_fit_covariance = curve_fit(double_gaussian,
                                                       vertex_z_bin_centers,
                                                       vertex_z_counts, 
                                                       p0=(amplitude_guess, -7, 2.5, amplitude_guess, -1.5, 1.5))
        y_fit = double_gaussian(vertex_z_bin_centers, *z_fit_parameters)
        z_fit_parameters_all_sectors.append(z_fit_parameters)
        axs[sector].plot(vertex_z_bin_centers, y_fit, color='r')
        axs[sector].set_xlabel("$v_{z}$ (cm)")
        axs[sector].set_title(f"Sector {sector+1}") 
        
    fig.tight_layout()
    fig.suptitle(plot_title+"\n Electron candidates", y=1.02)

In [ ]:
if target_vertex_selection:
    deuterium_mask_by_sector = []
    solid_mask_by_sector = []
    fig, axs = plt.subplots(3, 2, figsize=(18, 18))
    axs = axs.flatten()
    for sector in range(num_sectors):
        z_fit_parameters = z_fit_parameters_all_sectors[sector]
        print(z_fit_parameters)
        sector_cut = electrons["sector"]==(sector+1)
        
        vertex_z = np.array(electrons["v_z"][sector_cut][sf_mask_by_sector[sector]])
        vertex_z_counts, vertex_z_bins, _ = axs[sector].hist(
            vertex_z,
            bins=100,
            range=(-12,5),
            histtype='step'
        )
        vertex_z_bin_centers = (vertex_z_bins[:-1] + vertex_z_bins[1:])/2
        deuterium_z_mean = min(z_fit_parameters[1], z_fit_parameters[4])
        if deuterium_z_mean == z_fit_parameters[1]:
            deuterium_z_sigma = z_fit_parameters[2]
            solid_z_mean, solid_z_sigma = z_fit_parameters[4], z_fit_parameters[5]
        elif deuterium_z_mean == z_fit_parameters[4]:
            deuterium_z_sigma = z_fit_parameters[5]
            solid_z_mean, solid_z_sigma = z_fit_parameters[1], z_fit_parameters[2]
        
        # 2 sigma cut was chosen arbitrarily. Just a first attempt that looks OK.
        deuterium_cut = (vertex_z > (deuterium_z_mean - 2.5 * deuterium_z_sigma) ) &\
                        (vertex_z < (deuterium_z_mean + 2.5 * deuterium_z_sigma) )
        solid_cut     = (vertex_z > (solid_z_mean - 4 * solid_z_sigma) ) &\
                        (vertex_z < (solid_z_mean + 4 * solid_z_sigma) )
        deuterium_mask_by_sector.append(deuterium_cut)
        solid_mask_by_sector.append(solid_cut)
        
        axs[sector].hist(vertex_z[deuterium_cut],
             bins = 100,
             range=(-12, 5),
             color='b',
             label="Deuterium target",
             alpha=.8)
        axs[sector].hist(vertex_z[solid_cut],
                 bins = 100,
                 range=(-12, 5),
                 color='r',
                 label="Solid target",
                 alpha=.8)
        
        axs[sector].set_xlabel("$v_{z}$ (cm)")
        axs[sector].set_title(f"Sector {sector+1}") 
        axs[sector].legend(loc='upper left')
    fig.tight_layout()
    # fig.suptitle(plot_title+"\n Electron candidates", y=1.02)

In [ ]:
fig = plt.figure(figsize=(10,8))
plt.hist2d(np.array(event_builder_electrons["x"]), np.array(event_builder_electrons["Q2"]), bins=(100,100), range=[(0,1),(0,10)], norm=mcolors.LogNorm());
plt.xlabel("x")
plt.ylabel("$Q^{2}~(GeV^2)$")
plt.title(f"{plot_title}\n Event Builder Electrons")
plt.colorbar()

In [ ]:
x_by_sector = []
Q2_by_sector = []
W_by_sector = []
y_by_sector = []

for sector in range(num_sectors):
    sector_cut = electrons["sector"]==(sector+1)
    x_by_sector.append(electrons["x"][sector_cut][sf_mask_by_sector[sector]])
    Q2_by_sector.append(electrons["Q2"][sector_cut][sf_mask_by_sector[sector]])
    y_by_sector.append(electrons["y"][sector_cut][sf_mask_by_sector[sector]])
    W_by_sector.append(electrons["W"][sector_cut][sf_mask_by_sector[sector]])

In [ ]:
print(f"Percent of EB electrons remaining after SF vs Edep cut: {round(len(ak.flatten(x_by_sector))/len(original_event_builder_electrons['pid']), 3)*100}")

In [ ]:
print(len(original_event_builder_electrons['pid']))

In [ ]:
print(len(electrons["x"]))

In [ ]:
print(len(ak.flatten(x_by_sector)))

In [ ]:
fig = plt.figure(figsize=(10,8))

H, xedges, yedges, _ = plt.hist2d(ak.flatten(x_by_sector).to_numpy(), ak.flatten(Q2_by_sector).to_numpy(), bins=(100,100), range=[(0.01,1),(0,12)], norm=mcolors.LogNorm());
plt.colorbar()
plt.xlabel("x")
plt.ylabel("$Q^{2} ~(GeV^{2})$")
plt.title(f"{plot_title}")

In [ ]:
if target_vertex_selection:
    x_by_sector = []
    Q2_by_sector = []
    W_by_sector = []
    y_by_sector = []
    
    for sector in range(num_sectors):
        sector_cut = electrons["sector"]==(sector+1)
        
        target_mask = deuterium_mask_by_sector[sector] | solid_mask_by_sector[sector]

        x_by_sector.append(electrons["x"][sector_cut][sf_mask_by_sector[sector]][target_mask])
        Q2_by_sector.append(electrons["Q2"][sector_cut][sf_mask_by_sector[sector]][target_mask])
        y_by_sector.append(electrons["y"][sector_cut][sf_mask_by_sector[sector]][target_mask])
        W_by_sector.append(electrons["W"][sector_cut][sf_mask_by_sector[sector]][target_mask])

In [ ]:
masked_branches_by_sector = []
target_type_by_sector = []
for sector in range(num_sectors):
    sector_mask = electrons["sector"]==(sector+1)
    if target_vertex_selection:
        target_type = []
        for deuterium, solid in zip(deuterium_mask_by_sector[sector], solid_mask_by_sector[sector]):
            if deuterium:
                target_type.append("D2")
            elif solid:
                target_type.append(solid_name)
        target_type_by_sector.append(target_type)
        
        target_mask = deuterium_mask_by_sector[sector] | solid_mask_by_sector[sector]
        masked_branches = electrons[sector_mask][sf_mask_by_sector[sector]][target_mask]
    else:
        masked_branches = electrons[sector_mask][sf_mask_by_sector[sector]]
    masked_branches_by_sector.append(masked_branches)

In [ ]:
if target_vertex_selection:
    output_path = f"/media/miguel/Elements_2024/CLAS_data/elecron_candidates_with_targetcuts_withfiducialcuts_W2GeVcut{file_name.replace('ntuples', '')}"
else:
    output_path = f"/media/miguel/Elements_2024/CLAS_data/electron_candidates_notargetcuts_withfiducialcuts_W2GeVcut{file_name.replace('ntuples', '')}"

fields = masked_branches_by_sector[0].fields
fields_to_save = ["Q2", "chi2pid", "W", "x", "y", 'v_x', 'v_y', 'v_z', 'p_x', 'p_y', 'p_z', 'p', 'theta','theta_degrees','phi_degrees', 'phi', 'sector', 'chi2', 'NDF', 'track_charge', 'E_PCAL', 'E_ECIN', 'E_ECOUT', 'Nphe_HTCC', "PCAL_U", "PCAL_W","PCAL_V", 'DC_region1_x', 'DC_region1_y', 'DC_region1_z', 'DC_region1_edge', 'DC_region2_x', 'DC_region2_y', 'DC_region2_z', 'DC_region2_edge', 'DC_region3_x', 'DC_region3_y', 'DC_region3_z', 'DC_region3_edge']
output_dictionary = {}
print(fields)
for field in fields_to_save:
    print(field)
    output_dictionary[field] = np.concatenate([
        masked_branches_by_sector[sector][field]
        for sector in range(num_sectors)
    ])
if target_vertex_selection:
    output_dictionary["target"] = np.concatenate(target_type_by_sector)

In [ ]:
with ur.recreate(output_path) as file:
    file["electrons"] = output_dictionary